# Notebook Overview — Build Feature Vectors

## Purpose

This notebook builds final fixed-length feature vectors for both the **training** and **test** subsets by combining gradient-based, spatial, and frequency-domain feature CSV files into classifier-ready tables.

---

## Inputs

The notebook expects subset-specific feature CSV files:

* `metadata/features/train_gradient_features.csv`
* `metadata/features/train_spatial_features.csv`
* `metadata/features/train_frequency_features.csv`
* `metadata/features/test_gradient_features.csv`
* `metadata/features/test_spatial_features.csv`
* `metadata/features/test_frequency_features.csv`

Paths and constants are provided by:

* `src/project_config.py`

---

## Assumptions

* Feature extraction has already been completed for both subsets
* Each input CSV contains one row per image for its corresponding subset
* Metadata columns are repeated in each feature CSV
* Rows align across all three CSV files for the same images within each subset
* This notebook performs **feature-vector assembly only**
* No raw image access is required
* This notebook processes **both training and test subsets in a single run**

---

## Outputs

The following files are written under `metadata/vectors/`:

* `metadata/vectors/train_feature_vectors.csv`
* `metadata/vectors/test_feature_vectors.csv`

These files are written to the Colab runtime and are **not automatically persisted to Google Drive**.

---

## Final Feature Vector Structure

Each row contains:

* metadata columns:
  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* feature columns:
  * 8 gradient features
  * 9 spatial features
  * 8 frequency features

**Total DIP features: 25**

---

## Notes

* This notebook builds both training and test feature vectors in a **single execution**
* No normalization is performed here
* No model training is performed here
* Training and test data remain strictly separated:
  * training set → model development
  * test set → final evaluation only
* This notebook operates entirely on structured feature data (no image access required)
* All validation steps are **fail-fast** — execution stops immediately if inconsistencies are detected

---

**Next step:**  
Proceed to **06_Normalize_and_Prepare_Inputs.ipynb** to scale features and prepare classifier inputs.

---
---

### Step 1 — Environment Setup

* Clone the GitHub repository into the Colab runtime
* Import configuration from `project_config.py`
* Verify required feature CSV files exist

In [ ]:
# ============================================
# Step 1: Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_GRADIENT_FEATURES_PATH,
    TRAIN_SPATIAL_FEATURES_PATH,
    TRAIN_FREQUENCY_FEATURES_PATH,
    TEST_GRADIENT_FEATURES_PATH,
    TEST_SPATIAL_FEATURES_PATH,
    TEST_FREQUENCY_FEATURES_PATH,
    TRAIN_FEATURE_VECTOR_PATH,
    TEST_FEATURE_VECTOR_PATH,
    TRAIN_IMAGES,
    TEST_IMAGES,
    TRAIN_SUBSET,
    TEST_SUBSET,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
INPUT_FILES = [
    Path(TRAIN_GRADIENT_FEATURES_PATH),
    Path(TRAIN_SPATIAL_FEATURES_PATH),
    Path(TRAIN_FREQUENCY_FEATURES_PATH),
    Path(TEST_GRADIENT_FEATURES_PATH),
    Path(TEST_SPATIAL_FEATURES_PATH),
    Path(TEST_FREQUENCY_FEATURES_PATH),
]

TRAIN_OUTPUT_FILE = Path(TRAIN_FEATURE_VECTOR_PATH)
TEST_OUTPUT_FILE = Path(TEST_FEATURE_VECTOR_PATH)

OUTPUT_DIR = TRAIN_OUTPUT_FILE.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required feature CSV files...\n")

missing_files = [str(f) for f in INPUT_FILES if not f.exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing required feature files:\n{missing_files}"
    )

print("All required feature CSV files are present.")

if VERBOSE:
    print("\nInput files:")
    for f in INPUT_FILES:
        print(f"  - {f}")

    print(f"\nOutput files:")
    print(f"  - Train: {TRAIN_OUTPUT_FILE}")
    print(f"  - Test : {TEST_OUTPUT_FILE}")

print("\nStartup complete.")



### Step 2 — Configuration and Expected Sizes

* Define expected row counts for training and test subsets using configuration constants
* Display input and output file paths

In [ ]:
# ============================================
# Step 2: Define Expected Sizes and Display Paths
# ============================================

# -------------------------------------------------
# Expected row counts from project configuration
# -------------------------------------------------
EXPECTED_ROWS = {
    TRAIN_SUBSET: TRAIN_IMAGES,
    TEST_SUBSET: TEST_IMAGES,
}

# -------------------------------------------------
# Display configuration
# -------------------------------------------------
print("Training input files:")
print(f"  GRADIENT      = {TRAIN_GRADIENT_FEATURES_PATH}")
print(f"  SPATIAL       = {TRAIN_SPATIAL_FEATURES_PATH}")
print(f"  FREQUENCY     = {TRAIN_FREQUENCY_FEATURES_PATH}")
print(f"  OUTPUT        = {TRAIN_FEATURE_VECTOR_PATH}")
print(f"  EXPECTED ROWS = {EXPECTED_ROWS[TRAIN_SUBSET]}")

print("\nTest input files:")
print(f"  GRADIENT      = {TEST_GRADIENT_FEATURES_PATH}")
print(f"  SPATIAL       = {TEST_SPATIAL_FEATURES_PATH}")
print(f"  FREQUENCY     = {TEST_FREQUENCY_FEATURES_PATH}")
print(f"  OUTPUT        = {TEST_FEATURE_VECTOR_PATH}")
print(f"  EXPECTED ROWS = {EXPECTED_ROWS[TEST_SUBSET]}")

print("\nPath and expected-size configuration complete.")


### Step 3 — Load Feature CSVs

* Load gradient, spatial, and frequency feature CSV files for:
  * training subset
  * test subset

In [ ]:
# ============================================
# Step 3: Load Feature CSVs
# ============================================

train_paths = [
    TRAIN_GRADIENT_FEATURES_PATH,
    TRAIN_SPATIAL_FEATURES_PATH,
    TRAIN_FREQUENCY_FEATURES_PATH,
]

test_paths = [
    TEST_GRADIENT_FEATURES_PATH,
    TEST_SPATIAL_FEATURES_PATH,
    TEST_FREQUENCY_FEATURES_PATH,
]

# -------------------------------------------------
# Verify input files exist
# -------------------------------------------------
for path in train_paths + test_paths:
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing required file: {path}")

# -------------------------------------------------
# Load training feature CSVs
# -------------------------------------------------
df_grad_train = pd.read_csv(TRAIN_GRADIENT_FEATURES_PATH)
df_spat_train = pd.read_csv(TRAIN_SPATIAL_FEATURES_PATH)
df_freq_train = pd.read_csv(TRAIN_FREQUENCY_FEATURES_PATH)

# -------------------------------------------------
# Load test feature CSVs
# -------------------------------------------------
df_grad_test = pd.read_csv(TEST_GRADIENT_FEATURES_PATH)
df_spat_test = pd.read_csv(TEST_SPATIAL_FEATURES_PATH)
df_freq_test = pd.read_csv(TEST_FREQUENCY_FEATURES_PATH)

# -------------------------------------------------
# Display summary
# -------------------------------------------------
print("Training CSV shapes:")
print("  Gradient : ", df_grad_train.shape)
print("  Spatial  : ", df_spat_train.shape)
print("  Frequency: ", df_freq_train.shape)

print("\nTest CSV shapes:")
print("  Gradient : ", df_grad_test.shape)
print("  Spatial  : ", df_spat_test.shape)
print("  Frequency: ", df_freq_test.shape)

print("\nExpected rows:")
print("  Train:", EXPECTED_ROWS[TRAIN_SUBSET])
print("  Test :", EXPECTED_ROWS[TEST_SUBSET])


### Step 4 — Validate Row Counts and Alignment

For each subset:

* Confirm expected row counts
* Confirm the three feature tables align by length

In [ ]:
# ============================================
# Step 4: Validate Row Counts and Alignment
# ============================================

metadata_columns = ["filename", "source_dataset", "class_label", "subset"]

def validate_subset_alignment(df_grad, df_spat, df_freq, expected_rows, subset_name):
    # -------------------------------------------------
    # Validate row counts
    # -------------------------------------------------
    if len(df_grad) != expected_rows:
        raise ValueError(
            f"{subset_name} gradient CSV row count mismatch: expected {expected_rows}, got {len(df_grad)}"
        )

    if len(df_spat) != expected_rows:
        raise ValueError(
            f"{subset_name} spatial CSV row count mismatch: expected {expected_rows}, got {len(df_spat)}"
        )

    if len(df_freq) != expected_rows:
        raise ValueError(
            f"{subset_name} frequency CSV row count mismatch: expected {expected_rows}, got {len(df_freq)}"
        )

    # -------------------------------------------------
    # Validate metadata alignment
    # -------------------------------------------------
    for col in metadata_columns:
        if not df_grad[col].equals(df_spat[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and spatial CSVs in column: {col}"
            )
        if not df_grad[col].equals(df_freq[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and frequency CSVs in column: {col}"
            )

    if VERBOSE:
        print(f"PASS: {subset_name} input CSVs have the expected row count")
        print(f"PASS: {subset_name} metadata columns align across all three CSVs")


# -------------------------------------------------
# Validate training subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nRow count and alignment validation complete.")



### Step 5 — Define Metadata and Feature Columns

* Define:
  * metadata column list
  * gradient feature columns
  * spatial feature columns
  * frequency feature columns

In [ ]:
# ============================================
# Step 5: Define Metadata and Feature Columns
# ============================================

metadata_cols = metadata_columns

gradient_feature_cols = [
    "Mean Gradient",
    "Std Gradient",
    "Max Gradient",
    "Gradient Entropy",
    "Edge Density",
    "Orientation Mean",
    "Orientation Std",
    "Orientation Entropy"
]

spatial_feature_cols = [
    "Global Entropy",
    "Local Entropy Mean",
    "Local Entropy Std",
    "Intensity Mean",
    "Intensity Std",
    "Laplacian Variance",
    "Patch Variance Mean",
    "Patch Variance Std",
    "Noise Residual Energy"
]

frequency_feature_cols = [
    "Low Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std"
]

print("Metadata columns: ", len(metadata_cols))
print("Gradient features:", len(gradient_feature_cols))
print("Spatial features: ", len(spatial_feature_cols))
print("Frequency features:", len(frequency_feature_cols))
print("Total DIP features:",
      len(gradient_feature_cols)
    + len(spatial_feature_cols)
    + len(frequency_feature_cols))


### Step 6 — Verify Required Columns Exist

* Confirm all expected metadata and feature columns are present

In [ ]:
# ============================================
# Step 6: Verify Required Columns Exist
# ============================================

def check_columns(df, required_cols, df_name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")
    if VERBOSE:
        print(f"{df_name}: all required columns present")

# -------------------------------------------------
# Training set checks
# -------------------------------------------------
check_columns(df_grad_train, metadata_cols + gradient_feature_cols, "Train Gradient CSV")
check_columns(df_spat_train, metadata_cols + spatial_feature_cols, "Train Spatial CSV")
check_columns(df_freq_train, metadata_cols + frequency_feature_cols, "Train Frequency CSV")


# -------------------------------------------------
# Test set checks
# -------------------------------------------------
check_columns(df_grad_test, metadata_cols + gradient_feature_cols, "Test Gradient CSV")
check_columns(df_spat_test, metadata_cols + spatial_feature_cols, "Test Spatial CSV")
check_columns(df_freq_test, metadata_cols + frequency_feature_cols, "Test Frequency CSV")

print("\nColumn verification complete.")



### Step 7 — Verify Metadata Alignment Across CSVs

For each subset:

* Confirm metadata columns match across all feature CSVs
* Confirm row-by-row alignment

In [ ]:
# ============================================
# Step 7: Verify Metadata Alignment Across CSVs
# ============================================

def check_metadata_alignment(df_grad, df_spat, df_freq, subset_name):
    for col in metadata_cols:
        same_grad_spat = df_grad[col].equals(df_spat[col])
        same_grad_freq = df_grad[col].equals(df_freq[col])

        if VERBOSE:
            print(f"{subset_name} | {col}:")
            print(f"  Gradient vs Spatial:   {same_grad_spat}")
            print(f"  Gradient vs Frequency: {same_grad_freq}")

        if not same_grad_spat or not same_grad_freq:
            raise ValueError(f"{subset_name}: mismatch detected in metadata column: {col}")

    print(f"PASS: {subset_name} metadata columns align across CSVs")


# -------------------------------------------------
# Training set
# -------------------------------------------------
check_metadata_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Test set
# -------------------------------------------------
check_metadata_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)

print("\nMetadata alignment verification complete.")



### Step 8 — Verify Subset Labels

* Confirm training rows are labeled `train`
* Confirm test rows are labeled `test`

In [ ]:
# ============================================
# Step 8: Verify Subset Labels Match Expected Subsets
# ============================================

def check_subset_labels(df_grad, df_spat, df_freq, expected_subset_value):
    unique_subsets_grad = sorted(df_grad["subset"].dropna().unique())
    unique_subsets_spat = sorted(df_spat["subset"].dropna().unique())
    unique_subsets_freq = sorted(df_freq["subset"].dropna().unique())

    if VERBOSE:
        print(f"{expected_subset_value} | Gradient subset values: ", unique_subsets_grad)
        print(f"{expected_subset_value} | Spatial subset values:  ", unique_subsets_spat)
        print(f"{expected_subset_value} | Frequency subset values:", unique_subsets_freq)

    if unique_subsets_grad != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} gradient CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_grad}"
        )

    if unique_subsets_spat != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} spatial CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_spat}"
        )

    if unique_subsets_freq != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} frequency CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_freq}"
        )

    print(f"PASS: subset labels match {expected_subset_value}")


# -------------------------------------------------
# Training set
# -------------------------------------------------
check_subset_labels(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Test set
# -------------------------------------------------
check_subset_labels(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)

print("\nSubset label verification complete.")



### Step 9 — Build Feature Vector Tables

For both subsets:

* Retain metadata columns
* Append:
  * gradient features (8)
  * spatial features (9)
  * frequency features (8)

In [ ]:
# ============================================
# Step 9: Build Final Feature Vector DataFrames
# ============================================

# -------------------------------------------------
# Build training feature-vector table
# -------------------------------------------------
df_feature_vectors_train = pd.concat(
    [
        df_grad_train[metadata_cols],
        df_grad_train[gradient_feature_cols],
        df_spat_train[spatial_feature_cols],
        df_freq_train[frequency_feature_cols]
    ],
    axis=1
)

# -------------------------------------------------
# Build test feature-vector table
# -------------------------------------------------
df_feature_vectors_test = pd.concat(
    [
        df_grad_test[metadata_cols],
        df_grad_test[gradient_feature_cols],
        df_spat_test[spatial_feature_cols],
        df_freq_test[frequency_feature_cols]
    ],
    axis=1
)

# -------------------------------------------------
# Display summary
# -------------------------------------------------
print("Training feature vector shape:", df_feature_vectors_train.shape)
print("Test feature vector shape:", df_feature_vectors_test.shape)

if VERBOSE:
    print("\nSample training rows:")
    display(df_feature_vectors_train.head(3))

    print("\nSample test rows:")
    display(df_feature_vectors_test.head(3))



### Step 10 — Validate Final Feature Vectors

* Confirm row counts match expected values
* Confirm feature count is correct
* Confirm no missing values

In [ ]:
# ============================================
# Step 10: Validate Final Feature Vector Tables
# ============================================

expected_feature_count = (
    len(gradient_feature_cols) +
    len(spatial_feature_cols) +
    len(frequency_feature_cols)
)

def validate_feature_vector_table(df_feature_vectors, expected_rows, subset_name):
    actual_feature_count = len(df_feature_vectors.columns) - len(metadata_cols)

    print(f"{subset_name} | Expected feature count: {expected_feature_count}")
    print(f"{subset_name} | Actual feature count:   {actual_feature_count}")

    if actual_feature_count != expected_feature_count:
        raise ValueError(f"{subset_name}: feature count mismatch")

    if len(df_feature_vectors) != expected_rows:
        raise ValueError(
            f"{subset_name}: final row count mismatch: "
            f"expected {expected_rows}, got {len(df_feature_vectors)}"
        )

    missing_counts = df_feature_vectors.isnull().sum()
    missing_counts = missing_counts[missing_counts > 0]

    if len(missing_counts) > 0:
        raise ValueError(f"{subset_name}: missing values detected:\n{missing_counts}")

    if VERBOSE:
        print(f"\n{subset_name} | Class counts:")
        print(df_feature_vectors["class_label"].value_counts())

        print(f"\n{subset_name} | Source counts:")
        print(df_feature_vectors["source_dataset"].value_counts())

        print(f"\n{subset_name} | Subset counts:")
        print(df_feature_vectors["subset"].value_counts())

    print(f"\nPASS: {subset_name} feature vector table validated")


# -------------------------------------------------
# Validate training feature-vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test feature-vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)

print("\nFinal feature vector validation complete.")



### Step 11 — Save Feature Vectors

* Save:
  * `train_feature_vectors.csv`
  * `test_feature_vectors.csv`
* Verify files were written successfully

In [ ]:
# ============================================
# Step 11: Save Final Feature Vectors
# ============================================

# -------------------------------------------------
# Ensure output directory exists
# -------------------------------------------------
TRAIN_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Save training feature vectors
# -------------------------------------------------
df_feature_vectors_train.to_csv(TRAIN_OUTPUT_FILE, index=False)
if VERBOSE:
    print(f"Saved: {TRAIN_OUTPUT_FILE}")
    print("Train shape:", df_feature_vectors_train.shape)

# -------------------------------------------------
# Save test feature vectors
# -------------------------------------------------
df_feature_vectors_test.to_csv(TEST_OUTPUT_FILE, index=False)
if VERBOSE:
    print(f"Saved: {TEST_OUTPUT_FILE}")
    print("Test shape:", df_feature_vectors_test.shape)

# -------------------------------------------------
# Verify saved files
# -------------------------------------------------
if not TRAIN_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Train output file not created: {TRAIN_OUTPUT_FILE}")

if not TEST_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Test output file not created: {TEST_OUTPUT_FILE}")

print("\nFeature vectors saved and verified.")

